In [ ]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 18.6 MB/s eta 0:00:00


In [17]:
import pandas as pd
import urllib.parse

# 1. ЗАВАНТАЖЕННЯ ДАТАСЕТІВ
print("Завантаження датасетів...")
# Треки (2000 випадкових)
df_tracks = pd.read_csv('/content/spotify-tracks-dataset.csv').sample(n=2000, random_state=42)

# Подкасти (читаємо рівно 500 рядків і відразу зупиняємось)
df_podcasts = pd.read_csv('/content/top_podcasts.csv', nrows=500, on_bad_lines='skip')

ONTOLOGY_IRI = "http://www.semanticweb.org/oleksiirak/ontologies/2026/3/Music_Platform_Ontology"

def clean_uri(text):
    if pd.isna(text): return "Unknown"
    clean = str(text).replace(" ", "_").replace("\"", "").replace("'", "").replace("(", "").replace(")", "").replace("&", "And").replace(",", "").replace("-", "_")
    return urllib.parse.quote(clean)

# 2. ПОБУДОВА БАЗОВОЇ ІЄРАРХІЇ
xml_lines = [
    '<?xml version="1.0"?>',
    f'<rdf:RDF xmlns="{ONTOLOGY_IRI}/"',
    f'     xml:base="{ONTOLOGY_IRI}/"',
    '     xmlns:owl="http://www.w3.org/2002/07/owl#"',
    '     xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"',
    '     xmlns:xml="http://www.w3.org/XML/1998/namespace"',
    '     xmlns:xsd="http://www.w3.org/2001/XMLSchema#"',
    '     xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#"',
    f'     xmlns:mpo="{ONTOLOGY_IRI}#">',
    '',
    f'    <owl:Ontology rdf:about="{ONTOLOGY_IRI}"/>',
    '',
    '    ',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#AudioContent"/>',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Genre"/>',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Collection"/>',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Person"/>',
    '',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Track">',
    f'        <rdfs:subClassOf rdf:resource="{ONTOLOGY_IRI}#AudioContent"/>',
    '    </owl:Class>',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#PodcastEpisode">',
    f'        <rdfs:subClassOf rdf:resource="{ONTOLOGY_IRI}#AudioContent"/>',
    '    </owl:Class>',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Audiobook">',
    f'        <rdfs:subClassOf rdf:resource="{ONTOLOGY_IRI}#AudioContent"/>',
    '    </owl:Class>',
    '',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Album">',
    f'        <rdfs:subClassOf rdf:resource="{ONTOLOGY_IRI}#Collection"/>',
    '    </owl:Class>',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Playlist">',
    f'        <rdfs:subClassOf rdf:resource="{ONTOLOGY_IRI}#Collection"/>',
    '    </owl:Class>',
    f'    <owl:Class rdf:about="{ONTOLOGY_IRI}#Artist">',
    f'        <rdfs:subClassOf rdf:resource="{ONTOLOGY_IRI}#Person"/>',
    '    </owl:Class>',
    '',
    '    ',
    f'    <owl:ObjectProperty rdf:about="{ONTOLOGY_IRI}#belongsToAlbum"/>',
    f'    <owl:ObjectProperty rdf:about="{ONTOLOGY_IRI}#createsTrack"/>',
    f'    <owl:ObjectProperty rdf:about="{ONTOLOGY_IRI}#hasGenre"/>',
    f'    <owl:DatatypeProperty rdf:about="{ONTOLOGY_IRI}#hasTitle"/>',
    f'    <owl:DatatypeProperty rdf:about="{ONTOLOGY_IRI}#popularityScore"/>',
    ''
]

artists_seen = set()
albums_seen = set()
genres_seen = set()

# 3. ДОДАВАННЯ ТРЕКІВ
print("Генерація треків...")
for index, row in df_tracks.iterrows():
    track_name = str(row['track_name'])
    artist_name = str(row['artists']).split(';')[0]
    album_name = str(row['album_name'])
    genre_name = str(row['track_genre']).capitalize()
    popularity = int(row['popularity'])

    t_id = f"Track_{clean_uri(track_name)}_{index}"
    ar_id = f"Artist_{clean_uri(artist_name)}"
    al_id = f"Album_{clean_uri(album_name)}"
    g_id = f"Genre_{clean_uri(genre_name)}"

    if g_id not in genres_seen:
        xml_lines.append(f'    <owl:NamedIndividual rdf:about="{ONTOLOGY_IRI}#{g_id}"><rdf:type rdf:resource="{ONTOLOGY_IRI}#Genre"/></owl:NamedIndividual>')
        genres_seen.add(g_id)

    if al_id not in albums_seen:
        xml_lines.append(f'    <owl:NamedIndividual rdf:about="{ONTOLOGY_IRI}#{al_id}"><rdf:type rdf:resource="{ONTOLOGY_IRI}#Album"/></owl:NamedIndividual>')
        albums_seen.add(al_id)

    if ar_id not in artists_seen:
        xml_lines.append(f'    <owl:NamedIndividual rdf:about="{ONTOLOGY_IRI}#{ar_id}">')
        xml_lines.append(f'        <rdf:type rdf:resource="{ONTOLOGY_IRI}#Artist"/>')
        xml_lines.append(f'        <mpo:hasGenre rdf:resource="{ONTOLOGY_IRI}#{g_id}"/>')
        xml_lines.append(f'    </owl:NamedIndividual>')
        artists_seen.add(ar_id)

    safe_title = track_name.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")
    xml_lines.append(f'    <owl:NamedIndividual rdf:about="{ONTOLOGY_IRI}#{t_id}">')
    xml_lines.append(f'        <rdf:type rdf:resource="{ONTOLOGY_IRI}#Track"/>')
    xml_lines.append(f'        <mpo:belongsToAlbum rdf:resource="{ONTOLOGY_IRI}#{al_id}"/>')
    xml_lines.append(f'        <mpo:hasTitle rdf:datatype="http://www.w3.org/2001/XMLSchema#string">{safe_title}</mpo:hasTitle>')
    xml_lines.append(f'        <mpo:popularityScore rdf:datatype="http://www.w3.org/2001/XMLSchema#integer">{popularity}</mpo:popularityScore>')
    xml_lines.append(f'    </owl:NamedIndividual>')

    xml_lines.append(f'    <rdf:Description rdf:about="{ONTOLOGY_IRI}#{ar_id}">')
    xml_lines.append(f'        <mpo:createsTrack rdf:resource="{ONTOLOGY_IRI}#{t_id}"/>')
    xml_lines.append(f'    </rdf:Description>')

# 4. ДОДАВАННЯ ПОДКАСТІВ
print("Генерація подкастів...")
for index, row in df_podcasts.iterrows():
    # Розумний пошук колонки з назвою подкасту
    podcast_name = "Unknown Podcast"
    if 'title' in df_podcasts.columns: podcast_name = str(row['title'])
    elif 'name' in df_podcasts.columns: podcast_name = str(row['name'])
    elif 'show_name' in df_podcasts.columns: podcast_name = str(row['show_name'])
    elif 'episode_name' in df_podcasts.columns: podcast_name = str(row['episode_name'])
    # ВИПРАВЛЕНО: Звертаємось до df_podcasts.columns замість row.columns
    else: podcast_name = str(row.iloc[1] if len(df_podcasts.columns) > 1 else row.iloc[0])

    p_id = f"Podcast_{clean_uri(podcast_name)}_{index}"
    safe_p_title = podcast_name.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")

    xml_lines.append(f'    <owl:NamedIndividual rdf:about="{ONTOLOGY_IRI}#{p_id}">')
    xml_lines.append(f'        <rdf:type rdf:resource="{ONTOLOGY_IRI}#PodcastEpisode"/>')
    xml_lines.append(f'        <mpo:hasTitle rdf:datatype="http://www.w3.org/2001/XMLSchema#string">[Podcast] {safe_p_title}</mpo:hasTitle>')
    xml_lines.append(f'    </owl:NamedIndividual>')

# 5. ДОДАВАННЯ СИНТЕТИЧНИХ ДАНИХ (Аудіокниги та Плейлисти)
print("Додавання аудіокниг та плейлистів...")
xml_lines.append(f'    <owl:NamedIndividual rdf:about="{ONTOLOGY_IRI}#Audiobook_HarryPotter">')
xml_lines.append(f'        <rdf:type rdf:resource="{ONTOLOGY_IRI}#Audiobook"/>')
xml_lines.append(f'        <mpo:hasTitle rdf:datatype="http://www.w3.org/2001/XMLSchema#string">Harry Potter and the Sorcerer&apos;s Stone (Narrated)</mpo:hasTitle>')
xml_lines.append(f'    </owl:NamedIndividual>')

xml_lines.append(f'    <owl:NamedIndividual rdf:about="{ONTOLOGY_IRI}#Playlist_TopGlobal2026">')
xml_lines.append(f'        <rdf:type rdf:resource="{ONTOLOGY_IRI}#Playlist"/>')
xml_lines.append(f'        <mpo:hasTitle rdf:datatype="http://www.w3.org/2001/XMLSchema#string">Top Global Hits 2026</mpo:hasTitle>')
xml_lines.append(f'    </owl:NamedIndividual>')

xml_lines.append('</rdf:RDF>')

# Запис файлу
output_name = 'ULTIMATE_Music_Ontology_Phase2.rdf'
with open(output_name, 'w', encoding='utf-8') as f:
    f.write('\n'.join(xml_lines))

print(f"Ідеально! Файл '{output_name}' з двома датасетами успішно згенеровано.")

Завантаження датасетів...
Генерація треків...
Генерація подкастів...
Додавання аудіокниг та плейлистів...
Ідеально! Файл 'ULTIMATE_Music_Ontology_Phase2.rdf' з двома датасетами успішно згенеровано.
